# 05. DAgger を試し、GAIL の落とし穴を見る

**対応するテキスト**: [docs/06_DAggerを試す.md](../docs/06_DAggerを試す.md) ／ [docs/07_GAILと評価の落とし穴.md](../docs/07_GAILと評価の落とし穴.md)

**前提**: [04_bc_job.ipynb](04_bc_job.ipynb) が完了していること（BC の結果と比較するため）。

このノートブックで行うこと:

1. **DAgger** を 3 シードで実行し、BC と比べる
2. **⚠ 可変ホライズン環境で GAIL がエラーになることを、手元で実際に体験する**
3. **GAIL** を固定ホライズン環境で実行し、**結果を記録する**

> [!WARNING]
> **このノートブックの Azure ジョブは Azure 上で実行検証していません。**
> 本ハンズオンの構築時に検証したのは**ローカル実行だけ**です。
> Azure ジョブの所要時間・費用は記載していません。**あなたの環境で確認してください。**

> ⚠ **サブスクリプション ID を書き込んだノートブックをコミットしないでください。**

In [ ]:
# ============================================================
#  ここを自分の環境に書き換えてください（01・03・04 と同じ値）
# ============================================================
SUBSCRIPTION_ID = "<SUBSCRIPTION_ID>"
RESOURCE_GROUP = "<RESOURCE_GROUP>"
WORKSPACE_NAME = "<AML_WORKSPACE_NAME>"

COMPUTE_NAME = "cpu-cluster"
ENV_REF = "il-pickplace-env@latest"
DATA_ASSET_NAME = "il-pickplace-demos"
EXPERIMENT = "il-dagger-gail"
EXPERIMENT_BC = "il-bc"          # 04 で使った実験名（比較に使います）

TAGS = {
    "project": "il-workshop",
    "owner": "<your-alias>",
    "delete-after": "<YYYY-MM-DD>",
}

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import mlflow

ml_client = MLClient(
    credential=DefaultAzureCredential(exclude_interactive_browser_credential=False),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
ws = ml_client.workspaces.get(WORKSPACE_NAME)
print("接続しました:", ws.name)

tracking_uri = getattr(ws, "mlflow_tracking_uri", None)
if tracking_uri is None:
    tracking_uri = (
        f"azureml://{ws.location}.api.azureml.ms/mlflow/v1.0"
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )
mlflow.set_tracking_uri(tracking_uri)
print("MLflow 追跡先を設定しました。")

## 1. DAgger を実行する

**DAgger は、学習中の方策で環境を動かし、そこで訪れた状況について専門家に「正解」を聞き直します。**

```mermaid
flowchart LR
    A["いまの方策で動かす"] --> B["訪れた状況を集める"]
    B --> C["専門家に正解を問い合わせる"]
    C --> D["データに追加して学習し直す"]
    D --> A
```

> [!IMPORTANT]
> **だから DAgger には専門家本体が必要です。**
> 本ハンズオンの専門家は [../src/scripted_expert.py](../src/scripted_expert.py) の**コード**なので、
> ジョブのスナップショットに含まれて一緒に送られます。
> `imitation` の **`NonTrainablePolicy`**（手書き方策のための基底クラス）を継承しています。

In [ ]:
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes

DAGGER_COMMAND = (
    "python train_il.py"
    " --algo dagger"
    " --demos-dir ${{inputs.demos}}"
    " --output-dir ${{outputs.model}}"
    " --dagger-timesteps ${{inputs.dagger_timesteps}}"
    " --batch-size ${{inputs.batch_size}}"
    " --seed ${{inputs.seed}}"
)


def submit_dagger(seed, dagger_timesteps=20000, batch_size=32):
    job = command(
        code="../src",
        command=DAGGER_COMMAND,
        inputs=dict(
            demos=Input(type=AssetTypes.URI_FOLDER, path=f"azureml:{DATA_ASSET_NAME}@latest"),
            dagger_timesteps=dagger_timesteps,
            batch_size=batch_size,
            seed=seed,
        ),
        outputs=dict(model=Output(type=AssetTypes.URI_FOLDER)),
        environment=ENV_REF,
        compute=COMPUTE_NAME,
        experiment_name=EXPERIMENT,
        display_name=f"dagger_steps{dagger_timesteps}_seed{seed}",
        tags={**TAGS, "algo": "dagger"},
    )
    returned = ml_client.jobs.create_or_update(job)
    print(f"  投入: {returned.display_name}  ({returned.name})")
    return returned

import time


def wait_all(jobs, poll_seconds=30):
    # 投入した全ジョブが終了状態になるまで待つ
    pending = {job.name for job in jobs}
    while pending:
        finished = set()
        for name in sorted(pending):
            status = str(ml_client.jobs.get(name).status)
            if status in ("Completed", "Failed", "Canceled"):
                print(f"  {name}: {status}")
                finished.add(name)
        pending -= finished
        if pending:
            print(f"  ...残り {len(pending)} 本。{poll_seconds} 秒後に再確認します。")
            time.sleep(poll_seconds)
    print("すべて終了しました。")

In [ ]:
print("DAgger を投入します（3 シード）:")
dagger_jobs = [submit_dagger(seed=s) for s in (0, 1, 2)]

In [ ]:
wait_all(dagger_jobs)

## 2. BC と比べる

> [!WARNING]
> **⚠ 学習量をそろえずに比べてはいけません。**
> DAgger は **1 ラウンドごとに BC 学習を行う**ため、`--dagger-timesteps` だけでは
> **実際の勾配更新回数が分かりません。**
>
> **本ハンズオンでは、DAgger の更新回数を実測したうえで BC と同水準にそろえています。**
> 詳細は [docs/06_DAggerを試す.md](../docs/06_DAggerを試す.md) を参照してください。

In [ ]:
import pandas as pd

runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT, EXPERIMENT_BC], output_format="pandas"
)

COLS = {
    "tags.mlflow.runName": "run_name",
    "params.algo": "algo",
    "params.n_demo_episodes": "n_demos",
    "params.epochs": "epochs",
    "params.dagger_timesteps": "dagger_steps",
    "params.seed": "seed",
    "metrics.eval_success_rate": "success_rate",
    "metrics.eval_return_mean": "return_mean",
    "metrics.normalized_return": "normalized",
}
available = {k: v for k, v in COLS.items() if k in runs.columns}
df = runs[list(available)].rename(columns=available).dropna(subset=["success_rate"])
for col in ("n_demos", "seed"):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("=== 手法ごとの成績 ===")
print(df.groupby("algo")[["success_rate", "return_mean"]].agg(["count", "mean", "min", "max"]))
print()
df.sort_values(["algo", "seed"])

## 3. ⚠ 可変ホライズン環境で GAIL を動かしてみる（**ローカルで実行**）

**ここは Azure ではなく手元の PC で実行します。**

[02_explore_env.ipynb](02_explore_env.ipynb) で、**素の環境は成功した瞬間に終わる**ことを確認しました。
その環境で **GAIL を起動すると何が起きるか**を、実際に見ます。

> **期待される結果: `ValueError` で停止する。** これは**バグではなく、意図された安全装置**です。

In [ ]:
import sys

sys.path.insert(0, "../src")

import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy

from imitation.algorithms.adversarial.gail import GAIL
from imitation.data import rollout
from imitation.data.wrappers import RolloutInfoWrapper
from imitation.rewards.reward_nets import BasicRewardNet
from imitation.util.networks import RunningNorm
from imitation.util.util import make_vec_env

from pick_place_env import VARIABLE_HORIZON_ENV_ID
from scripted_expert import rollout_policy

rng = np.random.default_rng(0)
venv = make_vec_env(                      # ← 固定ホライズン化していない、素の環境
    VARIABLE_HORIZON_ENV_ID, rng=rng, n_envs=8,
    post_wrappers=[lambda env, _: RolloutInfoWrapper(env)],
)
try:
    demos = rollout.rollout(
        rollout_policy, venv,
        rollout.make_sample_until(min_timesteps=None, min_episodes=30),
        rng=rng,
    )
    print("収集したエピソードの長さ:", sorted({len(t) for t in demos}))
    print("→ 長さがそろっていないことを確認してください。\n")

    try:
        learner = PPO(env=venv, policy=MlpPolicy, seed=0, verbose=0)
        GAIL(
            demonstrations=demos, demo_batch_size=256,
            gen_replay_buffer_capacity=512, n_disc_updates_per_round=8,
            venv=venv, gen_algo=learner,
            reward_net=BasicRewardNet(
                observation_space=venv.observation_space,
                action_space=venv.action_space,
                normalize_input_layer=RunningNorm,
            ),
        ).train(16_384)
    except Exception as exc:
        print(f"停止しました: {type(exc).__name__}")
        print(exc)
    else:
        print("エラーになりませんでした（ライブラリの版により挙動が変わることがあります）。")
finally:
    venv.close()

### 何が起きたのか

**「終了条件が報酬の情報を漏らす」** —— [02_explore_env.ipynb](02_explore_env.ipynb) で確認したとおりです。
素の環境では「エピソードが短い＝成功」が確定情報なので、
**報酬を学習する手法はそれを見るだけで点を取れてしまいます。**

> [!WARNING]
> **`allow_variable_horizon=True` で回避してはいけません。**
> エラーは消えますが、**評価が意味を失います。** 固定ホライズン化した環境を使ってください。

> ⚠ **エラー メッセージ中の URL は実在しません。**
> 正しくは **https://imitation.readthedocs.io/en/latest/main-concepts/variable_horizon.html** です
> （[docs/07 の 7.4](../docs/07_GAILと評価の落とし穴.md)）。

## 4. GAIL を固定ホライズン環境で実行する（Azure ML）

固定ホライズン化した `il/PandaPickAndPlace-v0` なら、GAIL は**エラーにならずに動きます。**

> ⚠ **100,000 環境ステップを学習します。BC よりはるかに時間がかかります。**
> ローカルでは **443 秒**（BC の最良条件は約 197 秒）でした。

In [ ]:
GAIL_COMMAND = (
    "python train_il.py"
    " --algo gail"
    " --demos-dir ${{inputs.demos}}"
    " --output-dir ${{outputs.model}}"
    " --n-demo-episodes ${{inputs.n_demo_episodes}}"
    " --gail-timesteps ${{inputs.gail_timesteps}}"
    " --seed ${{inputs.seed}}"
)

gail_job = command(
    code="../src",
    command=GAIL_COMMAND,
    inputs=dict(
        demos=Input(type=AssetTypes.URI_FOLDER, path=f"azureml:{DATA_ASSET_NAME}@latest"),
        n_demo_episodes=200,
        gail_timesteps=100000,
        seed=0,
    ),
    outputs=dict(model=Output(type=AssetTypes.URI_FOLDER)),
    environment=ENV_REF,
    compute=COMPUTE_NAME,
    experiment_name=EXPERIMENT,
    display_name="gail_pick_and_place_seed0",
    tags={**TAGS, "algo": "gail"},
)

returned_gail = ml_client.jobs.create_or_update(gail_job)
print("ジョブ名 :", returned_gail.name)
print("studio  :", returned_gail.studio_url)

In [ ]:
wait_all([returned_gail], poll_seconds=60)

gail_run = mlflow.get_run(returned_gail.name)
print("\n=== GAIL の結果 ===")
for k in ("eval_success_rate", "eval_return_mean", "eval_return_std",
          "normalized_return", "random_mean", "expert_mean"):
    if k in gail_run.data.metrics:
        print(f"  {k:22s}: {gail_run.data.metrics[k]:.3f}")

## 5. 結果の解釈

**ローカルでの実測結果と、結論の書き方は
[docs/07_GAILと評価の落とし穴.md](../docs/07_GAILと評価の落とし穴.md) の 7.5〜7.7 にまとめています。**

> [!IMPORTANT]
> **「GAIL は使えない」とは書けません。**
> 書けるのは **「この環境・この設定・この計算量では、学習の開始を確認できなかった」** です。

## ✅ チェックリスト

- [ ] DAgger を 3 シードで実行した
- [ ] **DAgger と BC の学習量（勾配更新回数）の違いを理解した**
- [ ] **素の環境で GAIL がエラーになることを自分で確認した**
- [ ] エラー メッセージ中の URL が実在しないことを確認した
- [ ] GAIL を固定ホライズン環境で実行し、**結果を記録した**
- [ ] **「この環境・この設定・この計算量で」という限定を付けて結論を書いた**

---

**次へ**: [docs/08_Sweepで改善する.md](../docs/08_Sweepで改善する.md) → [06_sweep_compare_cleanup.ipynb](06_sweep_compare_cleanup.ipynb)